# GNN-RWKV Agent OS: Deep Brain Trainer
This notebook trains the custom GNN-RWKV architecture on the Hinglish Everyday Conversations dataset. 
Designed for Google Colab (GPU Recommended).

In [ ]:
# 1. Install Dependencies
!pip install datasets torch re

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import json
import re
from datasets import load_dataset
from google.colab import files

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Model Architecture (GNN-RWKV Hybrid)

In [ ]:
class GNNRWKVCell(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        self.w_receptance = nn.Linear(dim, dim)
        self.w_key = nn.Linear(dim, dim)
        self.w_value = nn.Linear(dim, dim)
        self.time_decay = nn.Parameter(torch.ones(dim) * -0.5)
        self.ln = nn.LayerNorm(dim)
        
    def forward(self, x, edge_index, h):
        r = torch.sigmoid(self.w_receptance(x))
        k = self.w_key(x)
        v = self.w_value(x)
        
        row, col = edge_index
        neighbor_kv = torch.zeros_like(k)
        if edge_index.shape[1] > 0:
            neighbor_kv.index_add_(0, row, k[col] * v[col])
            
        new_h = (torch.exp(self.time_decay) * h) + neighbor_kv
        out = r * self.ln(new_h)
        return out, new_h

class SocialNarrativeModel(nn.Module):
    def __init__(self, vocab_size, num_nodes, dim=64):
        super().__init__()
        self.word_emb = nn.Embedding(vocab_size, dim)
        self.node_emb = nn.Embedding(num_nodes, dim)
        self.cell = GNNRWKVCell(dim)
        self.head = nn.Linear(dim, vocab_size)

    def forward(self, word_ids, node_ids, edge_index, h):
        w_x = self.word_emb(word_ids)
        n_x = self.node_emb(node_ids)
        graph_out, new_h = self.cell(n_x, edge_index, h)
        world_state = graph_out.mean(dim=0).unsqueeze(0)
        combined = w_x + world_state
        logits = self.head(combined)
        return logits, new_h

## Data Loading & Tokenization

In [ ]:
print("Fetching dataset...")
ds = load_dataset('Abhishekcr448/Hinglish-Everyday-Conversations-1M', split='train[:50000]')

all_text = ""
for row in ds:
    all_text += row['input'] + " " + row['output'] + " "

all_text = re.sub(r'[^a-zA-Z\s]', '', all_text.lower())
words = all_text.split()
vocab = sorted(list(set(words)))
w2i = {w: i for i, w in enumerate(vocab)}
i2w = {i: w for w, i in enumerate(vocab)}
vocab_size = len(vocab)
data = [w2i[w] for w in words]

print(f"Vocab Size: {vocab_size} | Total Tokens: {len(data)}")

## Training Loop

In [ ]:
dim = 64
num_nodes = 5
epochs = 50 # Increase for better quality!
chunk_size = 500

model = SocialNarrativeModel(vocab_size, num_nodes, dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
edges = torch.tensor([[0, 1, 1, 3], [1, 0, 3, 1]], dtype=torch.long).to(device)
node_ids = torch.tensor([0, 1, 2, 3, 4]).to(device)

print("Starting Training...")
model.train()
for epoch in range(epochs):
    epoch_loss = 0
    chunks = 0
    for i in range(0, len(data) - chunk_size, chunk_size):
        h = torch.zeros((num_nodes, dim)).to(device)
        optimizer.zero_grad()
        
        inp = torch.tensor(data[i:i+chunk_size]).to(device)
        tar = torch.tensor(data[i+1:i+chunk_size+1]).to(device)
        
        logits, _ = model(inp, node_ids, edges, h)
        loss = F.cross_entropy(logits, tar)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        chunks += 1
    
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Avg Loss: {epoch_loss/chunks:.4f}")

print("Training Complete!")

## Save & Export

In [ ]:
# Save Weights
torch.save(model.cpu().state_dict(), 'social_brain.pth')

# Save Vocab
vocab_data = {"w2i": w2i, "i2w": {str(k): v for k, v in i2w.items()}}
with open('vocab.json', 'w') as f:
    json.dump(vocab_data, f)

print("Files Saved! Downloading now...")
files.download('social_brain.pth')
files.download('vocab.json')